In [ ]:
# Berlin Food Delivery ETA Prediction
## Feature Engineering

Goal: Turn raw, cleaned data into model-ready features:
- Real distance from coordinates (haversine)
- Datetime-derived features (hour, day of week, rush hour)
- Prep time before pickup
- Encoded categorical variables

In [58]:
import pandas as pd
import numpy as np

train = pd.read_csv('train_cleaned.csv')
test = pd.read_csv('test_cleaned.csv')

train.head()

,ID,Delivery_person_ID,Delivery_person_Age,Delivery_person_Ratings,Restaurant_latitude,Restaurant_longitude,Delivery_location_latitude,Delivery_location_longitude,Order_Date,Time_Orderd,Time_Order_picked,Weatherconditions,Road_traffic_density,Vehicle_condition,Type_of_order,Type_of_vehicle,multiple_deliveries,Festival,City,Time_taken(min)
0,0x4607,INDORES13DEL02,37.0,4.9,22.745049,75.892471,22.765049,75.912471,19-03-2022,11:30:00,11:45:00,Sunny,High,2,Snack,motorcycle,0.0,No,Urban,24.0
1,0xb379,BANGRES18DEL02,34.0,4.5,12.913041,77.683237,13.043041,77.813237,25-03-2022,19:45:00,19:50:00,Stormy,Jam,2,Snack,scooter,1.0,No,Metropolitian,33.0
2,0x5d6d,BANGRES19DEL01,23.0,4.4,12.914264,77.678400,12.924264,77.688400,19-03-2022,08:30:00,08:45:00,Sandstorms,Low,0,Drinks,motorcycle,1.0,No,Urban,26.0
3,0x7a6a,COIMBRES13DEL02,38.0,4.7,11.003669,76.976494,11.053669,77.026494,05-04-2022,18:00:00,18:10:00,Sunny,Medium,0,Buffet,motorcycle,1.0,No,Metropolitian,21.0
4,0x70a2,CHENRES12DEL01,32.0,4.6,12.972793,80.249982,13.012793,80.289982,26-03-2022,13:30:00,13:45:00,Cloudy,High,1,Snack,scooter,1.0,No,Metropolitian,30.0


In [59]:
## Haversine distance function

def haversine_distance(lat1, lon1, lat2, lon2):
    R = 6371  # Earth radius in km
    lat1, lon1, lat2, lon2 = map(np.radians, [lat1, lon1, lat2, lon2])
    dlat = lat2 - lat1
    dlon = lon2 - lon1
    a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return R * c

for df in [train, test]:
    df['distance_km'] = haversine_distance(
        df['Restaurant_latitude'], df['Restaurant_longitude'],
        df['Delivery_location_latitude'], df['Delivery_location_longitude']
    )

train['distance_km'].describe()

count    45593.000000
mean        99.303911
std       1099.731281
min          1.465067
25%          4.663493
50%          9.264281
75%         13.763977
max      19692.674606
Name: distance_km, dtype: float64

In [60]:
## checking for invalid distances

# Some rows had unrealistic distances (e.g. 0 km or 1000+ km due to bad coordinates)
print(train['distance_km'].sort_values(ascending=False).head(10))
print(train[train['distance_km'] < 0.5].shape[0], "rows under 0.5km")

33533    19692.674606
6788     19688.001288
2484     19683.687561
18826    19677.180552
9535     19070.408110
762      19070.337839
35535    19069.158946
22291    19068.246962
30633    19067.128547
43454    19066.150742
Name: distance_km, dtype: float64
0 rows under 0.5km


In [61]:
## handled distance outliers here

# Caped unrealistic distances at a sensible delivery-range max (e.g. 30km for a city)
distance_cap = 30
train['distance_km'] = train['distance_km'].clip(upper=distance_cap)
test['distance_km'] = test['distance_km'].clip(upper=distance_cap)

In [62]:
## parsed datetime features

for df in [train, test]:
    df['Order_Date'] = pd.to_datetime(df['Order_Date'], format='%d-%m-%Y')
    df['Time_Orderd'] = pd.to_datetime(df['Time_Orderd'], format='%H:%M:%S', errors='coerce').dt.time
    df['Time_Order_picked'] = pd.to_datetime(df['Time_Order_picked'], format='%H:%M:%S', errors='coerce').dt.time

    df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour
    df['day_of_week'] = df['Order_Date'].dt.dayofweek
    df['is_weekend'] = df['day_of_week'].isin([5, 6]).astype(int)
    df['is_rush_hour'] = df['order_hour'].isin([12, 13, 19, 20]).astype(int)

train[['order_hour', 'day_of_week', 'is_weekend', 'is_rush_hour']].head()

/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_15672/4240260406.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_15672/4240260406.py:8: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df['order_hour'] = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce').dt.hour


,order_hour,day_of_week,is_weekend,is_rush_hour
0,11.0,5,1,0
1,19.0,4,0,1
2,8.0,5,1,0
3,18.0,1,0,0
4,13.0,5,1,1


In [63]:
## preparation time before pickup (time between order placed and picked up)

for df in [train, test]:
    order_dt = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce')
    pickup_dt = pd.to_datetime(df['Time_Order_picked'].astype(str), errors='coerce')
    df['prep_time_min'] = (pickup_dt - order_dt).dt.total_seconds() / 60
    df['prep_time_min'] = df['prep_time_min'].apply(lambda x: x + 1440 if x < 0 else x)  # handle midnight rollover

train['prep_time_min'].describe()


/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_15672/4035421302.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['Time_Orderd'].astype(str), errors='coerce')
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_15672/4035421302.py:5: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  pickup_dt = pd.to_datetime(df['Time_Order_picked'].astype(str), errors='coerce')
/var/folders/cp/dhn31sd54_gfrh6szq15dn2r0000gn/T/ipykernel_15672/4035421302.py:4: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  order_dt = pd.to_datetime(df['Time_Orderd

count    43862.000000
mean         9.989399
std          4.087516
min          5.000000
25%          5.000000
50%         10.000000
75%         15.000000
max         15.000000
Name: prep_time_min, dtype: float64

In [66]:
## interaction features

# Traffic and weather often compound each other's effect on delay
traffic_map = {'Low': 1, 'Medium': 2, 'High': 3, 'Jam': 4}
train['traffic_level_num'] = train['Road_traffic_density'].map(traffic_map)
test['traffic_level_num'] = test['Road_traffic_density'].map(traffic_map)

train['distance_x_traffic'] = train['distance_km'] * train['traffic_level_num']
test['distance_x_traffic'] = test['distance_km'] * test['traffic_level_num']

train['is_bad_weather'] = train['Weatherconditions'].isin(['Stormy', 'Sandstorms', 'Fog']).astype(int)
test['is_bad_weather'] = test['Weatherconditions'].isin(['Stormy', 'Sandstorms', 'Fog']).astype(int)

In [67]:
## encoded remaining categoricals

categorical_cols = ['Type_of_order', 'Type_of_vehicle', 'City', 'Festival']
train_encoded = pd.get_dummies(train, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

train_encoded.shape, test_encoded.shape

((45593, 34), (11399, 33))

In [68]:
# Align test columns to match train (excluding the target)
target_col = 'Time_taken(min)'
train_cols = [c for c in train_encoded.columns if c != target_col]
test_encoded = test_encoded.reindex(columns=train_cols, fill_value=0)

print(train_encoded.shape, test_encoded.shape)


(45593, 34) (11399, 33)


In [69]:
## check the NaNs first

print("Missing prep_time_min:", train['prep_time_min'].isnull().sum())
print("As % of data:", round(train['prep_time_min'].isnull().sum() / len(train) * 100, 2), "%")

Missing prep_time_min: 0
As % of data: 0.0 %


In [64]:
## imputed with median

order_hour_median = train['order_hour'].median()
prep_median = train['prep_time_min'].median()

train['order_hour'] = train['order_hour'].fillna(order_hour_median)
test['order_hour'] = test['order_hour'].fillna(order_hour_median)
train['prep_time_min'] = train['prep_time_min'].fillna(prep_median)
test['prep_time_min'] = test['prep_time_min'].fillna(prep_median)

train['is_rush_hour'] = train['order_hour'].isin([12, 13, 19, 20]).astype(int)
test['is_rush_hour'] = test['order_hour'].isin([12, 13, 19, 20]).astype(int)

print("order_hour NaNs:", train['order_hour'].isnull().sum())
print("prep_time_min NaNs:", train['prep_time_min'].isnull().sum())

order_hour NaNs: 0
prep_time_min NaNs: 0


In [70]:
print(train['distance_km'].describe())

count    45593.000000
mean         9.926962
std          5.916366
min          1.465067
25%          4.663493
50%          9.264281
75%         13.763977
max         30.000000
Name: distance_km, dtype: float64


In [71]:
## save feature-engineered data

train_encoded.to_csv('train_features.csv', index=False)
test_encoded.to_csv('test_features.csv', index=False)
print("Saved train_features.csv and test_features.csv")

Saved train_features.csv and test_features.csv


In [72]:
import requests
import pandas as pd

url = "https://archive-api.open-meteo.com/v1/archive"
params = {
    "latitude": 52.52,
    "longitude": 13.405,
    "start_date": "2022-02-11",
    "end_date": "2022-04-06",
    "daily": ["temperature_2m_mean", "precipitation_sum", "windspeed_10m_max"],
    "timezone": "Europe/Berlin"
}

response = requests.get(url, params=params)
berlin_weather = pd.DataFrame(response.json()["daily"])
berlin_weather["time"] = pd.to_datetime(berlin_weather["time"])
berlin_weather = berlin_weather.rename(columns={
    "time": "Order_Date",
    "temperature_2m_mean": "berlin_temp_c",
    "precipitation_sum": "berlin_precip_mm",
    "windspeed_10m_max": "berlin_windspeed_max"
})

print(berlin_weather.shape)
berlin_weather.head()

(55, 4)


,Order_Date,berlin_temp_c,berlin_precip_mm,berlin_windspeed_max
0,2022-02-11,3.7,0.7,18.6
1,2022-02-12,0.2,0.0,16.8
2,2022-02-13,2.6,0.0,18.4
3,2022-02-14,6.9,0.0,22.3
4,2022-02-15,6.4,0.1,23.5


In [73]:
## merged into train/test by date

cols_to_drop = ['berlin_temp_c', 'berlin_precip_mm', 'berlin_windspeed_max']
train = train.drop(columns=[c for c in cols_to_drop if c in train.columns])
test = test.drop(columns=[c for c in cols_to_drop if c in test.columns])

train = train.merge(berlin_weather, on="Order_Date", how="left")
test = test.merge(berlin_weather, on="Order_Date", how="left")

print("Missing after merge:", train["berlin_temp_c"].isnull().sum())

Missing after merge: 0


In [74]:
## finding the real date range

print(train['Order_Date'].min())
print(train['Order_Date'].max())
print(train['Order_Date'].dt.to_period('M').value_counts().sort_index())

2022-02-11 00:00:00
2022-04-06 00:00:00
Order_Date
2022-02     7242
2022-03    31989
2022-04     6362
Freq: M, Name: count, dtype: int64


In [75]:
## Saved intermediate cleaned + feature-engineered data
print(train.shape, test.shape)
print(train.columns.tolist())

(45593, 32) (11399, 31)
['ID', 'Delivery_person_ID', 'Delivery_person_Age', 'Delivery_person_Ratings', 'Restaurant_latitude', 'Restaurant_longitude', 'Delivery_location_latitude', 'Delivery_location_longitude', 'Order_Date', 'Time_Orderd', 'Time_Order_picked', 'Weatherconditions', 'Road_traffic_density', 'Vehicle_condition', 'Type_of_order', 'Type_of_vehicle', 'multiple_deliveries', 'Festival', 'City', 'Time_taken(min)', 'distance_km', 'order_hour', 'day_of_week', 'is_weekend', 'is_rush_hour', 'prep_time_min', 'traffic_level_num', 'distance_x_traffic', 'is_bad_weather', 'berlin_temp_c', 'berlin_precip_mm', 'berlin_windspeed_max']


In [76]:
## Encoded categoricals 

categorical_cols = ['Type_of_order', 'Type_of_vehicle', 'City', 'Festival']

train_encoded = pd.get_dummies(train, columns=categorical_cols, drop_first=True)
test_encoded = pd.get_dummies(test, columns=categorical_cols, drop_first=True)

print(train_encoded.shape, test_encoded.shape)

(45593, 37) (11399, 36)


In [77]:
## Aligned test columns to match train

target_col = 'Time_taken(min)'
train_cols = [c for c in train_encoded.columns if c != target_col]
test_encoded = test_encoded.reindex(columns=train_cols, fill_value=0)

print(train_encoded.shape, test_encoded.shape)

(45593, 37) (11399, 36)


In [78]:
## checked the weather columns made it through correctly or not

weather_cols = ['berlin_temp_c', 'berlin_precip_mm', 'berlin_windspeed_max']
print(train_encoded[weather_cols].describe())
print("\nAny NaNs left?", train_encoded[weather_cols].isnull().sum().sum())

       berlin_temp_c  berlin_precip_mm  berlin_windspeed_max
count   45593.000000      45593.000000          45593.000000
mean        4.457237          0.661582             20.582427
std         2.838642          1.655833              8.701709
min        -1.000000          0.000000              8.700000
25%         1.900000          0.000000             14.700000
50%         4.700000          0.000000             18.600000
75%         7.000000          0.200000             24.000000
max         9.800000          7.600000             49.700000

Any NaNs left? 0


In [79]:
## Saved final feature set

train_encoded.to_csv('train_features.csv', index=False)
test_encoded.to_csv('test_features.csv', index=False)
print("Saved train_features.csv and test_features.csv")

Saved train_features.csv and test_features.csv


In [36]:
## Summary: Feature Engineering Complete

**Starting point:** Cleaned data from `01_data_cleaning_eda.ipynb` (45,593 train rows, 20 columns)

**Features engineered:**
    
- **distance_km** — Haversine distance calculated from restaurant/delivery coordinates 
  (straight-line proxy for travel distance, no routing API used). Capped at 30km after 
  discovering corrupted coordinates producing physically impossible distances (up to 19,000+ km).
      
- **Datetime features** — order_hour, day_of_week, is_weekend, is_rush_hour, extracted 
  from parsed order timestamps.
      
- **prep_time_min** — time between order placed and order picked up. ~3.8% missing values 
  (tied to missing Time_Orderd timestamps) imputed with median.
    
- **Interaction features** — distance_x_traffic (distance × traffic level) and 
  is_bad_weather (flag for Stormy/Sandstorm/Fog conditions), to help capture compounding 
  delay effects.
      
- **Berlin weather enrichment** — merged real historical Berlin weather (temperature, 
  precipitation, wind speed) via the Open-Meteo API, matched by order date. Initial merge 
  attempt assumed a March-only date range; investigation revealed the true range was 
  Feb 11 – Apr 6, 2022, which was corrected before re-merging (0 missing values in final merge).
      
- **Categorical encoding** — one-hot encoded Type_of_order, Type_of_vehicle, City, and 
  Festival; train/test columns explicitly aligned to guarantee matching feature sets.

**Final shapes:** train_encoded (45,593 × 37), test_encoded (11,399 × 36)

**Output:** `train_features.csv`, `test_features.csv` — ready for modeling in `03_modeling.ipynb`

### Note on Berlin weather enrichment (for presentation)
The delivery data itself is not literally sourced from Berlin — it's a generic food-delivery 
dataset with dates spanning Feb–Apr 2022. We enrich it with **real historical Berlin weather** 
matched by date, to add a genuine local dimension and demonstrate external API data 
enrichment as a skill. This is presented as an illustrative "what if this were Berlin" 
enrichment, not a claim that the deliveries physically occurred in Berlin.

SyntaxError: invalid decimal literal (3361257971.py, line 3)